# Thief Detector
## This task tests your Image Processing skills to build a motion detection algorithm that alarms you when you have an unwanted visitor in your home.

## Steps
- 1. Get the live video feed from your webcam
- 2. Fix a scene (the place you want to monitor) and store it as a reference background image
    - Store the first frame as the reference background frame
- 3. For every frame, check if there is any unwanted object inside the scene you are monitoring
    - Use **Background Subtraction** concept (**cv2.absdiff( )**)
        - Subtract the current frame from the reference background image(frame) to see the changes in the scene
        - If there is enormous amount of pixels distrubed in the subtraction result image
            - unwanted visitor (place is unsafe --> alarm the authorities)
        - If there is no enormous amount of pixels distrubed in the subtraction result image
            - no unwanted visitor (place is safe)
- 4. Output the text **"UNSAFE"** in **red** color on the top right of the frame when there is an intruder in the scene.
- 5. Save the live feed
- 6. Submit the (.ipynb) file

## Get live video feed from webcam [10 points]

In [4]:
import numpy as np
import cv2


threshold_value = 25    
min_area = 1500         
# Create a VideoCapture object (0 is the in-built webcam of the laptop)
video_capture = cv2.VideoCapture(0)
if not video_capture.isOpened():
    raise RuntimeError("Could not open webcam")

# Get current width and height of frame
width = int(video_capture.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video_capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
print("the webcam is open, the picture size is", width, "x", height)

the webcam is open, the picture size is 640 x 480


## Read first frame, convert to Grayscale and store it as reference background image [10 points]

In [5]:

for i in range(30):
    video_capture.read()


ret, first_frame = video_capture.read()


gray_background = cv2.cvtColor(first_frame, cv2.COLOR_BGR2GRAY)


gray_background = cv2.GaussianBlur(gray_background, (21, 21), 0)


cv2.imshow('Background', gray_background)
cv2.waitKey(0)
cv2.destroyAllWindows()

## Compute Absolute Difference between Current and First frame [20 points]

In [6]:

ret, frame = video_capture.read()

gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
gray = cv2.GaussianBlur(gray, (21, 21), 0)


diff = cv2.absdiff(gray_background, gray)

print("the biggest change in this frame is", diff.max())


cv2.imshow('Background', gray_background)
cv2.imshow('Current frame', gray)
cv2.imshow('Difference', diff)
cv2.waitKey(0)
cv2.destroyAllWindows()

the biggest change in this frame is 46


## Apply threshold [5 points]

In [7]:

ret2, thresh = cv2.threshold(diff, threshold_value, 255, cv2.THRESH_BINARY)


thresh = cv2.dilate(thresh, None, iterations=2)


cv2.imshow('Threshold', thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()

## Find contours [10 points]

In [8]:

contours, hierarchy = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)

print("number of contours found:", len(contours))
for c in contours:
    print("area =", cv2.contourArea(c))

number of contours found: 11
area = 301.0
area = 16.0
area = 224.0
area = 125.5
area = 140.0
area = 422.5
area = 217.5
area = 293.0
area = 447.5
area = 36.5
area = 61.0


## Check if contourArea is large and draw rectangle around the object, output "UNSAFE" text in red color [30 points]

In [9]:

motion = 0
for c in contours:
    if cv2.contourArea(c) < min_area:
        continue
    motion = 1
    (x, y, w, h) = cv2.boundingRect(c)
    cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)


if motion == 1:
    cv2.putText(frame, 'UNSAFE', (width - 140, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
else:
    cv2.putText(frame, 'SAFE', (width - 110, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

print("motion =", motion)


cv2.imshow('Thief Detector', frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

motion = 0


## Display images [10 points]

In [ ]:

fourcc = cv2.VideoWriter_fourcc(*'MJPG')
out = cv2.VideoWriter('thief_detector_output.avi', fourcc, 20.0, (width, height))

old_motion = -1     # I use this so I print only when the answer changes

try:
    while True:
        
        ret, frame = video_capture.read()
        if ret == False:
            break

     
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (21, 21), 0)

        diff = cv2.absdiff(gray_background, gray)
        ret2, thresh = cv2.threshold(diff, threshold_value, 255,
                                     cv2.THRESH_BINARY)
        thresh = cv2.dilate(thresh, None, iterations=2)
        contours, hierarchy = cv2.findContours(thresh.copy(),
                                               cv2.RETR_EXTERNAL,
                                               cv2.CHAIN_APPROX_SIMPLE)

        motion = 0
        for c in contours:
            if cv2.contourArea(c) < min_area:
                continue
            motion = 1
            (x, y, w, h) = cv2.boundingRect(c)
            cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)

        if motion == 1:
            cv2.putText(frame, 'UNSAFE', (width - 140, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2)
            cv2.putText(frame, 'Motion detected (1)', (10, height - 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        else:
            cv2.putText(frame, 'SAFE', (width - 110, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)
            cv2.putText(frame, 'No motion detected (0)', (10, height - 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        
        if motion != old_motion:
            if motion == 1:
                print("1 - Motion detected, UNSAFE")
            else:
                print("0 - No motion detected, safe")
            old_motion = motion

       
        out.write(frame)

        
        cv2.imshow('Thief Detector', frame)
        cv2.imshow('GrayScale', gray)
        cv2.imshow('Difference', diff)
        cv2.imshow('Threshold', thresh)

        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'):     
            break
        if key == ord('r'):    
            gray_background = gray
            print("new background picture saved")

except KeyboardInterrupt:
   
    print("stopped")

print("the loop is finished, now run the last cell")

1 - Motion detected, UNSAFE


## Release objects [5 points]

In [ ]:
# When everything done, release the capture
video_capture.release()
out.release()
cv2.destroyAllWindows()
print("everything is closed and the video is saved as thief_detector_output.avi")